# Занятие 6. Итоговая проверка на test

Прогноз оттока «СвязьРегион». Ноутбук загружает модель, выбранную и сохранённую на этапе 5, и **без обучения** сравнивает её с правилом долга на отложенном периоде 01.09.2026. Затем делает один разбор результата, показывает, на какие признаки модель опирается, и проверяет повторяемость расчёта.

**Запуск** из корня репозитория, одной командой:

```
jupyter nbconvert --to notebook --execute --inplace lesson6/evaluate.ipynb
```

Ядро работает в папке `lesson6/`, все пути ниже относительны неё. Результаты — в `lesson6/output/`. Пути к входам и заранее принятые решения — `lesson6/config.json`. Расчёт начинается от подготовленных признаков этапа 4; исходные выгрузки заново не собираются.

Метрики и загрузка — функции комплекта (`starter/campaign_metrics.py`, `starter/evaluation_utils.py`). Файлы комплекта не изменяются.

| Шаг ноутбука | Шаг задания |
|---|---|
| 0. Конфигурация и протокол | 1 |
| 1. Модель этапа 5 и перенос validation | 2 (вход) |
| 2. Test: модель и baseline | 2 |
| 3. Разбор по сегментам и три признака | 3 |
| 4. Три проверки и повторный запуск | 4 |

---

## Шаг 0. Конфигурация и протокол

Протокол заполнен **10.11.2026, до первого расчёта на test**. Тогда же, до test, выбраны вариант разбора (сегменты «долг есть / долга нет») и три признака для объяснения: оба решения записаны в `config.json`, чтобы их нельзя было подогнать под результат.

| Что фиксируем | Значение |
|---|---|
| Выбранная на этапе 5 модель | `hgb_leaf4_nodig`: `HistGradientBoostingClassifier`, 4 листа, 17 признаков без цифрового источника; файл `lesson5/output/model/candidate.joblib`. Выбрана по заранее записанному правилу — максимальный validation Recall@K: 18 найденных из 63, у второго места 17 |
| Период проверки и исход | Test T = 01.09.2026, 00:00 МСК. Исход — расторжение договора в календарном сентябре, `[01.09.2026, 01.10.2026)`; выгрузка покрывает весь горизонт |
| Правило сравнения | Одна активная база; `K = floor(0,10 × N)`; равные score — по `customer_id` по возрастанию. Baseline — долг предыдущего месяца (`debt_prev_rub`) по убыванию; неизвестный долг — после всех известных, как на этапе 5 |
| Что измеряем | Recall@K и Precision@K; рядом N, K, найдено / ушли. Validation — из этапа 5, test — из текущего расчёта |
| Использовали ли test для выбора модели? | Нет. На этапах 4–5 строки test не загружались ни в одну модель и ни в одну метрику. Baseline на test считался на этапе 1 для критерия успеха в уставе (46,2%) — до появления моделей, на выбор он не влиял |

Модель проверяется как сохранена: без `fit`, без переобучения на train + validation, без подбора по test.

In [1]:
import json
import platform
import sys
from datetime import date
from math import comb
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

if not Path("evaluate.ipynb").exists():
    raise RuntimeError(f"Kernel must run in lesson6/, current folder is {Path.cwd()}")

STARTER = Path("starter")
sys.path.insert(0, str(STARTER.resolve()))
from campaign_metrics import KEY, campaign_metrics
from evaluation_utils import load_holdout, permutation_recall, predict_risk, segment_metrics

config = json.loads(Path("config.json").read_text(encoding="utf-8"))
PREPARED = Path(config["prepared"])
MODEL_PATH = Path(config["model"])
LESSON5 = Path(config["lesson5_output"])
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}
print("; ".join(f"{name} {version}" for name, version in VERSIONS.items()))

python 3.13.18; pandas 2.3.3; numpy 2.5.4; scikit-learn 1.9.1; joblib 1.6.0


In [2]:
CHECKS = []


def require(condition: bool, message: str) -> None:
    # Fail fast on a violated precondition.
    if not condition:
        raise ValueError(message)


def record_check(name: str, passed: bool, detail: str) -> None:
    # Checks are collected into output/checks.csv; a failed one stops the notebook in the last cell.
    CHECKS.append({"check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'ok' if passed else 'FAILED'}] {name} — {detail}")


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    frame.to_csv(path, sep=";", encoding="utf-8-sig", index=False, lineterminator="\n")


def baseline_score(X: pd.DataFrame, feature: str) -> np.ndarray:
    # Debt of the previous month; unknown debt ranks after every known value (a position, not an imputed amount).
    debt = X[feature]
    return debt.fillna(debt.min() - 1).to_numpy()


def contacts(ranked: pd.DataFrame) -> pd.DataFrame:
    # The contact list as it is delivered: keys, score, place in the common ranking, membership in top-K.
    return ranked[KEY + ["score", "rank", "selected"]].reset_index(drop=True)

*Шаг 0. Конфигурация и протокол*

---

## Шаг 1. Модель этапа 5 и перенос validation

Прежде чем открывать test, нужно убедиться, что загружен именно тот объект, который выбирали на этапе 5, и что входные таблицы те же.

- ID опыта в файле модели совпадает с `lesson5/output/selection.json`; там же записано, что test на этапе 5 не использовался.
- Контрольные суммы `features.csv`, `labels.csv`, `split.csv` совпадают с суммами, сохранёнными в модели при обучении.
- Загруженная модель без `fit` воспроизводит validation-прогнозы этапа 5: score с допуском 1e-12, ранги и состав top-K — точно.

Отпечаток состояния Pipeline (`joblib.hash`) снимается сразу после загрузки. В конце ноутбука он сравнивается снова: если бы где-то произошло обучение, отпечаток изменился бы.

In [3]:
bundle = joblib.load(MODEL_PATH)
pipeline = bundle["pipeline"]
FRACTION = bundle["config"]["campaign_fraction"]
BASELINE_FEATURE = bundle["config"]["baseline_feature"]
MODEL_ID = bundle["selected_experiment"]
STATE_AT_LOAD = joblib.hash(pipeline)
print(f"Модель: {MODEL_ID}; обучена на scikit-learn {bundle['versions']['scikit-learn']}; "
      f"признаков: {len(bundle['numeric_columns']) + len(bundle['categorical_columns'])}")

selection = json.loads((LESSON5 / "selection.json").read_text(encoding="utf-8"))
record_check("the loaded model is the candidate selected at lesson 5, and lesson 5 did not use test",
             MODEL_ID == selection["selected_experiment"] and selection["test_used"] is False
             and selection["refit_on_train_plus_validation"] is False,
             f"{MODEL_ID}; test_used = {selection['test_used']}")

import hashlib
current_sums = {name: hashlib.sha256((PREPARED / name).read_bytes()).hexdigest() for name in bundle["data_version"]}
record_check("input tables are the ones the model was trained on", current_sums == bundle["data_version"],
             "; ".join(f"{name} {digest[:12]}" for name, digest in current_sums.items()))
record_check("scikit-learn version equals the version the model was saved with",
             sklearn.__version__ == bundle["versions"]["scikit-learn"], f"{sklearn.__version__}")

Модель: hgb_leaf4_nodig; обучена на scikit-learn 1.9.1; признаков: 17
[ok] the loaded model is the candidate selected at lesson 5, and lesson 5 did not use test — hgb_leaf4_nodig; test_used = False
[ok] input tables are the ones the model was trained on — features.csv 85d558c06d06; labels.csv 1054edfb6907; split.csv e68d40faaf60
[ok] scikit-learn version equals the version the model was saved with — 1.9.1


In [4]:
validation = load_holdout(PREPARED, bundle, "validation", config["observation_end_exclusive"])
_, validation_model, validation_ranked = campaign_metrics(validation["keys"], validation["y"],
                                                          predict_risk(pipeline, validation["X"]), FRACTION)
_, validation_baseline, _ = campaign_metrics(validation["keys"], validation["y"],
                                             baseline_score(validation["X"], BASELINE_FEATURE), FRACTION)

saved = pd.read_csv(LESSON5 / "validation_predictions.csv", sep=";", encoding="utf-8-sig",
                    dtype={"customer_id": str, "snapshot_at": str})
paired = contacts(validation_ranked).merge(saved, on=KEY, validate="one_to_one", suffixes=("", "_lesson5"))
record_check("the loaded model reproduces the lesson 5 validation predictions without fit",
             len(paired) == len(saved) == len(validation_ranked)
             and bool(np.allclose(paired.score, paired.score_lesson5, rtol=0, atol=1e-12))
             and (paired["rank"] == paired.rank_lesson5).all() and (paired.selected == paired.selected_lesson5).all(),
             f"{len(paired)} rows; max |score difference| = {(paired.score - paired.score_lesson5).abs().max():.1e}; ranks and top-K equal")

# Validation rows of the quality table are carried over from lesson 5 and cross-checked against this recomputation.
lesson5_metrics = read_csv(LESSON5 / "metrics_by_date.csv")
lesson5_validation = lesson5_metrics[lesson5_metrics.part == "validation"].set_index("id")
carried = {"baseline": validation_baseline, MODEL_ID: validation_model}
record_check("validation metrics of lesson 5 are carried over unchanged",
             all(int(lesson5_validation.loc[i, c]) == int(table[c].iloc[0]) for i, table in carried.items()
                 for c in ("n_active", "K", "n_churn", "hits")),
             "; ".join(f"{i}: {int(t.hits.iloc[0])} of {int(t.n_churn.iloc[0])}, K = {int(t.K.iloc[0])}" for i, t in carried.items()))

[ok] the loaded model reproduces the lesson 5 validation predictions without fit — 1929 rows; max |score difference| = 9.9e-17; ranks and top-K equal
[ok] validation metrics of lesson 5 are carried over unchanged — baseline: 14 of 63, K = 192; hgb_leaf4_nodig: 18 of 63, K = 192


*Шаг 1. Модель этапа 5 и перенос validation*

---

## Шаг 2. Test: модель и baseline

Единственное обращение к test. Признаки и метки сентября соединяются по ключу `(customer_id, snapshot_at)`; в расчёт входит вся активная база на 01.09.2026, включая строки с пропусками. Загрузчик комплекта останавливается, если исход какой-либо строки неизвестен или горизонт выходит за границу выгрузки: неизвестный исход нулём не заменяется.

Модель выдаёт score через `predict_proba` класса 1 — метод `fit` не вызывается. Для baseline score — долг предыдущего месяца. Оба списка строит одна функция `campaign_metrics`: сортировка по score по убыванию, равные — по `customer_id`, первые `K = floor(0,10 × N)`.

In [5]:
test = load_holdout(PREPARED, bundle, "test", config["observation_end_exclusive"])
require(set(test["keys"].snapshot_at.str.slice(0, 10)) == {config["test_date"]}, "test must be the single protocol date")

test_score = predict_risk(pipeline, test["X"])
_, test_model, test_ranked = campaign_metrics(test["keys"], test["y"], test_score, FRACTION)
_, test_baseline, test_baseline_ranked = campaign_metrics(test["keys"], test["y"],
                                                          baseline_score(test["X"], BASELINE_FEATURE), FRACTION)

base_check = read_csv(PREPARED / "check_active_base.csv")
base_check = base_check[base_check.snapshot_at.str.startswith(config["test_date"])].iloc[0]
record_check("test base is the whole active base of the date and its keys are unique",
             len(test["y"]) == int(base_check.n_independent) and int(test["y"].sum()) == int(base_check.churn_independent)
             and not test["keys"].duplicated(KEY).any(),
             f"N = {len(test['y'])}, churned = {int(test['y'].sum())}; equal to the independent recount of lesson 4")
record_check("model and baseline are compared on the same base with the same K",
             test_model[["snapshot_at", "n_active", "K", "n_churn"]].equals(test_baseline[["snapshot_at", "n_active", "K", "n_churn"]])
             and set(map(tuple, test_ranked[KEY].values)) == set(map(tuple, test_baseline_ranked[KEY].values)),
             f"N = {int(test_model.n_active.iloc[0])}, K = {int(test_model.K.iloc[0])} for both")
record_check("baseline on test reproduces the charter value",
             round(100 * test_baseline.recall_at_K.iloc[0], 1) == config["charter_baseline_test_recall_percent"],
             f"{int(test_baseline.hits.iloc[0])} of {int(test_baseline.n_churn.iloc[0])} = {test_baseline.recall_at_K.iloc[0]:.1%}; "
             f"charter (lesson 1): {config['charter_baseline_test_recall_percent']}%")


def quality_row(period: str, approach: str, table: pd.DataFrame) -> dict:
    row = table.iloc[0]
    return {"period": period, "approach": approach, "snapshot_at": row.snapshot_at[:10], "n_active": int(row.n_active),
            "K": int(row.K), "hits": int(row.hits), "n_churn": int(row.n_churn),
            "recall_at_K": row.recall_at_K, "precision_at_K": row.precision_at_K, "average_precision": row.average_precision}


quality = pd.DataFrame([quality_row("validation", "baseline", validation_baseline),
                        quality_row("validation", "model", validation_model),
                        quality_row("test", "baseline", test_baseline), quality_row("test", "model", test_model)])
view = pd.DataFrame({"Период": quality.period.str.capitalize(), "Подход": quality.approach.map({"baseline": "Baseline", "model": "Модель"}),
                     "N / K": quality.n_active.astype(str) + " / " + quality.K.astype(str),
                     "Найдено / ушли": quality.hits.astype(str) + " / " + quality.n_churn.astype(str),
                     "Recall@K": quality.recall_at_K.map("{:.1%}".format), "Precision@K": quality.precision_at_K.map("{:.1%}".format)})
view

[ok] test base is the whole active base of the date and its keys are unique — N = 1916, churned = 65; equal to the independent recount of lesson 4
[ok] model and baseline are compared on the same base with the same K — N = 1916, K = 191 for both
[ok] baseline on test reproduces the charter value — 30 of 65 = 46.2%; charter (lesson 1): 46.2%


,Период,Подход,N / K,Найдено / ушли,Recall@K,Precision@K
0,Validation,Baseline,1929 / 192,14 / 63,22.2%,7.3%
1,Validation,Модель,1929 / 192,18 / 63,28.6%,9.4%
2,Test,Baseline,1916 / 191,30 / 65,46.2%,15.7%
3,Test,Модель,1916 / 191,30 / 65,46.2%,15.7%


### Насколько списки различаются

Списки модели и правила сравниваются поимённо. Перевес проверяется тем же точным знаковым тестом, что на этапе 5: по ушедшим, которых нашёл только один из подходов.

In [6]:
def compare_lists(model_ranked: pd.DataFrame, rule_ranked: pd.DataFrame) -> dict:
    model_top, rule_top = model_ranked[model_ranked.selected], rule_ranked[rule_ranked.selected]
    model_found = set(model_top.customer_id[model_top.target == 1])
    rule_found = set(rule_top.customer_id[rule_top.target == 1])
    only_model, only_rule = len(model_found - rule_found), len(rule_found - model_found)
    discordant = only_model + only_rule
    p = min(1.0, 2 * sum(comb(discordant, i) for i in range(min(only_model, only_rule) + 1)) / 2 ** discordant) if discordant else 1.0
    return {"K": len(model_top), "ушли": int(model_ranked.target.sum()), "нашёл baseline": len(rule_found),
            "нашла модель": len(model_found), "разница": len(model_found) - len(rule_found),
            "общих абонентов в списках": len(set(model_top.customer_id) & set(rule_top.customer_id)),
            "ушедших нашли оба": len(model_found & rule_found), "только модель": only_model, "только baseline": only_rule,
            "p (знаковый тест)": round(p, 3)}


validation_baseline_ranked = campaign_metrics(validation["keys"], validation["y"],
                                              baseline_score(validation["X"], BASELINE_FEATURE), FRACTION)[2]
lists = pd.DataFrame({"validation 01.08.2026": compare_lists(validation_ranked, validation_baseline_ranked),
                      "test 01.09.2026": compare_lists(test_ranked, test_baseline_ranked)})
lists

,validation 01.08.2026,test 01.09.2026
K,192.000,191.0
ушли,63.000,65.0
нашёл baseline,14.000,30.0
нашла модель,18.000,30.0
разница,4.000,0.0
общих абонентов в списках,162.000,167.0
ушедших нашли оба,14.000,28.0
только модель,4.000,2.0
только baseline,0.000,2.0
p (знаковый тест),0.125,1.0


**На test модель и правило долга нашли поровну: по 30 уходов из 65**, Recall@K 46,2%, Precision@K 15,7% при K = 191.

- Преимущество с validation (+4 найденных) на test не сохранилось: разница 0.
- Списки совпадают на 167 абонентов из 191. Из ушедших 28 нашли оба подхода, 2 — только модель, 2 — только правило.
- Критерий успеха из устава — Recall@K на test **выше** 46,2% — не выполнен: модель сравнялась с правилом, но не обошла его.
- У самого правила результат от августа к сентябрю вырос с 22,2% до 46,2%. Разброс между месяцами больше любой разницы между моделью и правилом, поэтому один месяц не доказывает ни преимущества, ни его отсутствия.

Это согласуется с этапом 5: на проверочных датах внутри train та же конфигурация тоже не обходила правило (92 против 97 из 236).

*Шаг 2. Test: модель и baseline*

---

## Шаг 3. Разбор по сегментам и три признака

### Одно сегментное сравнение: долг есть / долга нет

Вариант выбран до расчёта на test. Причина: правило долга по построению не может выбрать абонента без долга, пока должников больше K, — а их больше K на каждой дате. Значит, все ушедшие без долга для правила недостижимы. Вопрос к модели: находит ли она кого-то из них, или её список — те же должники в другом порядке.

Группы определяются по `debt_prev_rub` на T. Попадание берётся из **общего** top-K: отдельные 10% внутри группы не отбираются.

In [7]:
SEGMENT = config["segment"]
group_names = SEGMENT["groups"]
debt = test["X"][SEGMENT["feature"]]
require(debt.notna().all(), "unknown debt on test needs its own group")
groups = test["keys"].assign(group=np.where(debt > 0, group_names["positive"], group_names["zero"]))

by_model = segment_metrics(test_ranked, groups, SEGMENT["name"]).set_index("group")
by_rule = segment_metrics(test_baseline_ranked, groups, SEGMENT["name"]).set_index("group")
segments = pd.DataFrame({
    "n": by_model.n, "n_churn": by_model.n_churn, "churn_rate": by_model.n_churn / by_model.n,
    "baseline_selected": by_rule.n_selected, "baseline_hits": by_rule.hits,
    "model_selected": by_model.n_selected, "model_hits": by_model.hits,
}).loc[[group_names["positive"], group_names["zero"]]].reset_index()
record_check("segment table covers the whole test base and the common top-K",
             segments.n.sum() == len(test["y"]) and segments.n_churn.sum() == int(test["y"].sum())
             and segments.model_selected.sum() == segments.baseline_selected.sum() == int(test_model.K.iloc[0])
             and segments.model_hits.sum() == int(test_model.hits.iloc[0])
             and segments.baseline_hits.sum() == int(test_baseline.hits.iloc[0]),
             f"groups {segments.n.tolist()}, churned {segments.n_churn.tolist()}, selected sum = K")
validation_top = validation_ranked[validation_ranked.selected].merge(
    validation["keys"].assign(debt=validation["X"][SEGMENT["feature"]].to_numpy()), on=KEY, validate="one_to_one")
print(f"Для справки, validation: абонентов без долга в списке модели — {int((validation_top.debt <= 0).sum())} из {len(validation_top)}")
segments.round(4)

[ok] segment table covers the whole test base and the common top-K — groups [426, 1490], churned [40, 25], selected sum = K
Для справки, validation: абонентов без долга в списке модели — 0 из 192


,group,n,n_churn,churn_rate,baseline_selected,baseline_hits,model_selected,model_hits
0,долг > 0,426,40,0.0939,191,30,191,30
1,долга нет,1490,25,0.0168,0,0,0,0


**Модель не выходит за пределы группы должников.**

| Группа на 01.09.2026 | Абонентов | Ушли | Baseline: в списке / найдено | Модель: в списке / найдено |
|---|---:|---:|---|---|
| Долг > 0 | 426 | 40 (9,4%) | 191 / 30 | 191 / 30 |
| Долга нет | 1 490 | 25 (1,7%) | 0 / 0 | 0 / 0 |

- Оба списка целиком состоят из должников: 191 из 191. Модель не выбрала ни одного абонента без долга — как и на validation. Она переставляет должников внутри той же группы, а не находит новых уходящих.
- 25 ушедших из 65 (38%) не имели долга. Их не находит ни правило, ни модель: в имеющихся признаках нет сигнала, который поднял бы их выше должников.
- Потолок для любого ранжирования внутри должников — 40 из 65, то есть 61,5%. Оба подхода взяли 30 из этих 40.

Ограничение разбора: одна дата, 40 и 25 событий в группах. Различие долей ухода между группами (9,4% против 1,7%) повторяет вывод EDA на train (7,5% против 1,4%); это связь, а не причина ухода.

### На какие признаки опирается модель

Готовых коэффициентов у бустинга нет, поэтому используется перестановочная важность из комплекта — **на validation, не на test**. Значения одного признака перемешиваются между абонентами внутри даты, остальные остаются на месте; считается, на сколько падает Recall@K. Среднее по 30 перестановкам. На validation 63 ухода, так что падение на 1,6 п.п. — это один найденный уход: меньшие значения от нуля не отличить.

Три признака для объяснения выбраны до test: два с наибольшим падением и стаж — признак, на который рассчитывал план этапа 4.

In [8]:
PERMUTATION = config["permutation"]
require(PERMUTATION["part"] == "validation", "permutation importance must not touch test")
importance = permutation_recall(pipeline, validation, FRACTION, PERMUTATION["repeats"], PERMUTATION["seed"]).reset_index(drop=True)
importance["mean_hits_lost"] = importance.mean_recall_drop * int(validation["y"].sum())
train_link = read_csv(PREPARED / "train_feature_target_link.csv").set_index("feature").spearman_with_target.astype(float)
importance["train_spearman_with_target"] = importance.feature.map(train_link)
importance["explained"] = importance.feature.isin(config["explained_features"])
require(set(config["explained_features"]) <= set(importance.feature), "explained features must be model inputs")
importance[["feature", "mean_recall_drop", "std_recall_drop", "mean_hits_lost", "train_spearman_with_target", "explained"]].round(4)

,feature,mean_recall_drop,std_recall_drop,mean_hits_lost,train_spearman_with_target,explained
0,debt_prev_rub,0.1534,0.0438,9.6667,0.1707,True
1,debt_change_rub,0.0254,0.0172,1.6000,0.1324,True
2,traffic_gb_mean_3m,0.0143,0.0132,0.9000,-0.0021,False
3,debt_months_3m,0.0074,0.0098,0.4667,0.1078,False
4,payment_delay_prev_days,0.0053,0.0095,0.3333,0.1320,False
5,traffic_change_gb,0.0053,0.0118,0.3333,-0.0195,False
6,support_billing_count_3m,0.0053,0.0075,0.3333,-0.0107,False
7,tenure_days,0.0042,0.0147,0.2667,-0.0057,True
8,traffic_per_day_3m,0.0000,0.0000,0.0000,-0.0012,False
9,support_count_3m,0.0000,0.0000,0.0000,0.0033,False


| Признак | Что измеряет и почему может быть связан с уходом | Основание |
|---|---|---|
| `debt_prev_rub` | Долг на конец предыдущего месяца. Неоплаченный счёт — частый спутник решения уйти: абонент перестал платить за услугу, которой не собирается пользоваться | EDA (H1): у должников уходит 7,5%, у остальных 1,4%, различие есть во всех 19 срезах. Перестановка на validation: Recall@K падает на 15,3 п.п. — теряется около 10 из 18 найденных |
| `debt_change_rub` | Изменение долга за последний месяц (долг M−1 минус долг M−2). Отличает новую неоплату от старого долга, который абонент носит давно | Ранговая корреляция с уходом на train 0,13. Перестановка: −2,5 п.п., около 1,6 найденного ухода — на границе различимого |
| `tenure_days` | Стаж абонента в днях. Новые абоненты могли ещё не привыкнуть к услуге и уходят легче | EDA (H4): «данных недостаточно» — 3,6% / 3,4% / 2,7% по группам стажа, интервал разницы включает ноль. Перестановка: −0,4 п.п., меньше одного ухода — модель этот признак практически не использует |

Модель держится на одном признаке — том же, по которому сортирует правило. Перестановка любого из остальных 16 признаков стоит не больше одного-двух найденных уходов, а у 8 признаков число найденных не меняется совсем. Отсюда и равенство с baseline: по сути модель — сортировка по долгу с небольшими поправками.

Связь в данных не доказывает причину ухода. Перестановочная важность посчитана на одном месяце с 63 уходами и показывает, на что опирается эта модель, а не что влияет на абонентов.

*Шаг 3. Разбор по сегментам и три признака*

---

## Шаг 4. Три проверки и повторный запуск

1. **Сопоставимость** — ключи и активная база сохранены, у модели и baseline одинаковые N и K. Проверено на шаге 2.
2. **Корректность оценки** — загружена выбранная модель (шаг 1); отпечаток состояния Pipeline после всех расчётов равен отпечатку сразу после загрузки, то есть ни подготовка, ни классификатор не обучались. В коде ноутбука нет ни одного вызова `fit`.
3. **Повторный запуск** — модель загружается с диска второй раз и на тех же входах даёт тот же список контактов и те же метрики. Дополнительно результат сравнивается с файлами предыдущего запуска ноутбука, если они есть: состав и порядок списка и таблица качества должны совпасть. Сравниваются ранги и попадание в K, а не байты файлов: последние разряды score могут отличаться между машинами, на список это не влияет.

Повтор подтверждает воспроизводимость расчёта, а не устойчивость качества на новых данных.

In [9]:
record_check("no fitting happened: the Pipeline state after all scoring equals the state at load",
             joblib.hash(pipeline) == STATE_AT_LOAD, f"joblib.hash = {STATE_AT_LOAD[:12]}")

repeat_bundle = joblib.load(MODEL_PATH)
_, repeat_metrics, repeat_ranked = campaign_metrics(test["keys"], test["y"], predict_risk(repeat_bundle["pipeline"], test["X"]), FRACTION)
record_check("reloading the same model on the same inputs gives the same contact list and metrics",
             contacts(repeat_ranked).equals(contacts(test_ranked)) and repeat_metrics.equals(test_model),
             f"{int(test_ranked.selected.sum())} contacts, found {int(repeat_metrics.hits.iloc[0])} of {int(repeat_metrics.n_churn.iloc[0])}")

FILES = {"model": OUTPUT / "test_model_predictions.csv", "baseline": OUTPUT / "test_baseline_predictions.csv",
         "quality": OUTPUT / "quality_table.csv"}
LIST_COLUMNS = KEY + ["rank", "selected"]
COUNT_COLUMNS = ["period", "approach", "snapshot_at", "n_active", "K", "hits", "n_churn"]
if all(path.exists() for path in FILES.values()):
    previous_lists_equal = all(
        pd.read_csv(FILES[name], sep=";", encoding="utf-8-sig", dtype={"customer_id": str, "snapshot_at": str})[LIST_COLUMNS]
        .equals(contacts(ranked)[LIST_COLUMNS]) for name, ranked in (("model", test_ranked), ("baseline", test_baseline_ranked)))
    previous_quality = pd.read_csv(FILES["quality"], sep=";", encoding="utf-8-sig", dtype={"snapshot_at": str})
    record_check("contact lists and the quality table equal the previous run of this notebook",
                 previous_lists_equal and previous_quality[COUNT_COLUMNS].equals(quality[COUNT_COLUMNS]),
                 "both test lists (rank, selected) and all counts of the quality table are unchanged")
else:
    record_check("contact lists and the quality table equal the previous run of this notebook", True,
                 "first run: nothing to compare with")

[ok] no fitting happened: the Pipeline state after all scoring equals the state at load — joblib.hash = a1708055e948
[ok] reloading the same model on the same inputs gives the same contact list and metrics — 191 contacts, found 30 of 65
[ok] contact lists and the quality table equal the previous run of this notebook — both test lists (rank, selected) and all counts of the quality table are unchanged


Запись результатов. Списки контактов содержат ключи, score, rank и признак попадания в K; будущей метки в них нет — она остаётся в `lesson4/output/labels.csv` и нужна только для оценки.

In [10]:
write_csv(contacts(test_ranked), FILES["model"])
write_csv(contacts(test_baseline_ranked), FILES["baseline"])
write_csv(quality.round(6), FILES["quality"])
write_csv(lists.T.rename_axis("period").reset_index(), OUTPUT / "list_comparison.csv")
write_csv(segments.round(6), OUTPUT / "segment_debt.csv")
write_csv(importance.round(6), OUTPUT / "permutation_validation.csv")
run_info = {"run_date": date.today().isoformat(), "model": MODEL_ID, "model_file": MODEL_PATH.as_posix(),
            "pipeline_state": STATE_AT_LOAD, "versions": VERSIONS, "fit_calls": 0}
(OUTPUT / "run_info.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

checks = pd.DataFrame(CHECKS)
write_csv(checks, OUTPUT / "checks.csv")
failed = checks[~checks.passed]
require(failed.empty, f"{len(failed)} check(s) failed: {failed.check.tolist()}")
print(f"Все проверки пройдены: {len(checks)}; дата запуска {run_info['run_date']}")

Все проверки пройдены: 12; дата запуска 2026-11-10


*Шаг 4. Три проверки и повторный запуск*

---

## Итог

| Период | Подход | N / K | Найдено / ушли | Recall@K | Precision@K |
|---|---|---|---|---:|---:|
| Validation 01.08.2026 | Baseline | 1 929 / 192 | 14 / 63 | 22,2% | 7,3% |
| Validation 01.08.2026 | Модель | 1 929 / 192 | 18 / 63 | 28,6% | 9,4% |
| Test 01.09.2026 | Baseline | 1 916 / 191 | 30 / 65 | 46,2% | 15,7% |
| Test 01.09.2026 | Модель | 1 916 / 191 | 30 / 65 | 46,2% | 15,7% |

- На отложенном месяце модель нашла столько же уходов, сколько правило долга. Преимущество с validation не подтвердилось.
- Модель выбирает только должников и опирается на тот же признак, что и правило; 38% ушедших (без долга) не находит ни один подход.
- Модель загружена без обучения, список и метрики при повторной загрузке совпали.

Рекомендация и ограничения — в `README.md`. Файлы: `output/quality_table.csv`, `output/test_model_predictions.csv`, `output/test_baseline_predictions.csv`, `output/segment_debt.csv`, `output/permutation_validation.csv`, `output/checks.csv`.